In [1]:
# @title 1. Instalar Dependências e LivePortrait
!git clone https://github.com/KwaiVGI/LivePortrait
%cd LivePortrait
!pip install -r requirements.txt
!pip install firebase-admin moviepy

Cloning into 'LivePortrait'...
remote: Enumerating objects: 1092, done.
remote: Counting objects: 100% (311/311), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 1092 (delta 271), reused 246 (delta 246), pack-reused 781 (from 3)
Receiving objects: 100% (1092/1092), 38.77 MiB | 20.55 MiB/s, done.
Resolving deltas: 100% (567/567), done.
/content/LivePortrait
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 311.7 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 858.3 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.5/881.5 kB 2.1 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 3.0 MB/s eta 0:00:00a 0:00:01
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This

In [ ]:
# @title 2. Conectar ao Firebase
import firebase_admin
from firebase_admin import credentials, firestore, storage
import os
import time
import requests
from subprocess import run

# Caminho para o seu arquivo JSON de credenciais
cred_path = '/content/the-symbiotic-protocol-firebase-adminsdk-fbsvc-0effdcdda6.json'

if not firebase_admin._apps:
    cred = credentials.Certificate(cred_path)
    firebase_admin.initialize_app(cred, {
        'storageBucket': 'the-symbiotic-protocol.firebasestorage.app'
    })

db = firestore.client()
bucket = storage.bucket()

print("✅ Conectado ao Firebase!")


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/fabri/Downloads/TSP/the-symbiotic-protocol-firebase-adminsdk-fbsvc-0effdcdda6.json'

In [ ]:
# @title 3. Iniciar Worker (LivePortrait)
def process_avatar(job_id, photo_url, audio_url):
    print(f"🚀 Processando Job: {job_id}")
    
    # 1. Download dos arquivos
    photo_path = "input_photo.jpg"
    audio_path = "input_audio.mp3"
    
    with open(photo_path, "wb") as f:
        f.write(requests.get(photo_url).content)
    with open(audio_path, "wb") as f:
        f.write(requests.get(audio_url).content)
        
    # 2. Executar LivePortrait
    # Nota: Usamos o script de inferência padrão adaptado para áudio
    # Para simplicidade, usamos o comando de CLI
    output_dir = f"animations/{job_id}"
    os.makedirs(output_dir, exist_ok=True)
    
    print("🎬 Animando...")
    # Comando simplificado (ajuste conforme a documentação do LivePortrait para áudio)
    # Por padrão o LivePortrait usa vídeo de referência, mas existem wrappers para áudio
    # Usaremos um comando placeholder que exemplifica a chamada
    !python inference.py --source $photo_path --driving $audio_path --output_dir $output_dir
    
    # 3. Upload do resultado
    result_video = os.path.join(output_dir, "result.mp4")
    if os.path.exists(result_video):
        blob = bucket.blob(f"avatars/{job_id}.mp4")
        blob.upload_from_filename(result_video)
        blob.make_public()
        
        # 4. Atualizar Firestore
        db.collection('avatar_jobs').document(job_id).update({
            'status': 'completed',
            'video_url': blob.public_url
        })
        print(f"✅ Job {job_id} Concluído!")
    else:
        print(f"❌ Erro ao gerar vídeo para {job_id}")
        db.collection('avatar_jobs').document(job_id).update({'status': 'failed'})

# Loop Infinito
print("🕵️ Worker Ninja em busca de jobs...")
while True:
    jobs = db.collection('avatar_jobs').where('status', '==', 'pending').stream()
    for job in jobs:
        data = job.to_dict()
        process_avatar(job.id, data['photo_url'], data['audio_url'])
    
    time.sleep(5) # Espera 5 segundos antes de checar de novo
